In [ ]:
# ============================================================
# CELL 1: CÀI ĐẶT THƯ VIỆN
# ============================================================
import os
import time
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import torchvision.models as models
import torchvision.transforms as transforms
import faiss

print("✅ Import thư viện thành công!")
print(f"   PyTorch version: {torch.__version__}")
print(f"   Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")


In [4]:
# ============================================================
# CELL 2: CẤU HÌNH ĐƯỜNG DẪN
# Lưu ý: notebook nằm trong notebooks/ nên phải lùi 1 cấp
# ============================================================
BASE_DIR    = Path(D:\New folder (3)\project\data\raw)                          # gốc project
DATA_DIR    = BASE_DIR / 'data' / 'images'        # thư mục ảnh
CSV_PATH    = BASE_DIR / 'data' / 'train.csv'     # file metadata
RESULTS_DIR = BASE_DIR / 'results'                # lưu biểu đồ & metric
FEATURES_DIR= BASE_DIR / 'data' / 'features'     # lưu FAISS index

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE  = 224
TOP_K       = 5
MAX_IMAGES  = 3000   # tập con để chạy baseline
NUM_QUERY   = 100    # số ảnh dùng để test

print(f"✅ Cấu hình đường dẫn xong!")
print(f"   DATA_DIR    : {DATA_DIR}")
print(f"   CSV_PATH    : {CSV_PATH}")
print(f"   RESULTS_DIR : {RESULTS_DIR}")


SyntaxError: invalid syntax (3497108680.py, line 5)

In [ ]:
# ============================================================
# CELL 3 (A): ĐỌC VÀ KIỂM TRA DỮ LIỆU
# Lưu: không cần lưu file, kết quả in ra notebook
# ============================================================
df = pd.read_csv(CSV_PATH)

print("=" * 55)
print("THÔNG TIN CƠ BẢN DATASET")
print("=" * 55)
print(f"Số dòng (ảnh)    : {df.shape[0]:,}")
print(f"Số cột           : {df.shape[1]}")
print(f"Tên các cột      : {list(df.columns)}")
print()
print("Giá trị NULL:")
print(df.isnull().sum())
print()
print(f"Số dòng trùng lặp: {df.duplicated().sum()}")
print()
print("Thống kê label_group:")
label_counts = df['label_group'].value_counts()
print(f"  Tổng số nhóm           : {label_counts.shape[0]:,}")
print(f"  Số ảnh TB mỗi nhóm     : {label_counts.mean():.2f}")
print(f"  Nhóm ít ảnh nhất       : {label_counts.min()} ảnh")
print(f"  Nhóm nhiều ảnh nhất    : {label_counts.max()} ảnh")
print(f"  Nhóm có đúng 2 ảnh     : {(label_counts == 2).sum():,} nhóm")


In [ ]:
# ============================================================
# CELL 4 (A): BẢNG MÔ TẢ DỮ LIỆU
# ============================================================
desc = {
    "Tên bộ dữ liệu"        : "Shopee — Price Match Guarantee",
    "Nguồn"                  : "Kaggle 2021 (Shopee Competition)",
    "Số mẫu gốc"             : f"{df.shape[0]:,} ảnh",
    "Cột nhãn"               : "label_group",
    "Tổng số nhóm label"     : f"{label_counts.shape[0]:,} nhóm",
    "Số ảnh TB mỗi nhóm"    : f"{label_counts.mean():.1f} ảnh",
    "Loại bài toán"          : "Truy xuất ảnh tương đồng (CBIR)",
    "Metric đánh giá"        : "Precision@K, Recall@K, mAP",
}
df_desc = pd.DataFrame(list(desc.items()), columns=["Thuộc tính", "Giá trị"])
print(df_desc.to_string(index=False))


In [ ]:
# ============================================================
# CELL 5 (A): LỌC ẢNH LỖI + CHỌN TẬP CON
# Lưu: data/processed/subset.csv
# ============================================================
print("Đang kiểm tra ảnh tồn tại trong thư mục...")

valid_rows = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Kiểm tra ảnh"):
    img_path = DATA_DIR / row['image']
    if img_path.exists():
        valid_rows.append(row)

df_valid = pd.DataFrame(valid_rows).reset_index(drop=True)
print(f"\n✅ Ảnh hợp lệ: {len(df_valid):,} / {len(df):,}")
print(f"❌ Ảnh bị lỗi/thiếu: {len(df) - len(df_valid):,}")

# Chỉ giữ nhóm có ≥ 2 ảnh (cần thiết để tính Recall)
group_size = df_valid['label_group'].value_counts()
valid_groups = group_size[group_size >= 2].index
df_filtered = df_valid[df_valid['label_group'].isin(valid_groups)].reset_index(drop=True)
print(f"\nSau khi lọc nhóm có ≥ 2 ảnh: {len(df_filtered):,} ảnh")

# Chọn tập con cân bằng MAX_IMAGES ảnh
df_subset = (df_filtered
             .groupby('label_group', group_keys=False)
             .apply(lambda x: x.sample(min(len(x), 5), random_state=42))
             .sample(min(MAX_IMAGES, len(df_filtered)), random_state=42)
             .reset_index(drop=True))

print(f"✅ Tập con cuối: {len(df_subset):,} ảnh, "
      f"{df_subset['label_group'].nunique():,} nhóm")

# Lưu tập con
processed_dir = BASE_DIR / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)
df_subset.to_csv(processed_dir / 'subset.csv', index=False)
print(f"💾 Đã lưu: data/processed/subset.csv")


In [ ]:
# ============================================================
# CELL 6 (A): BIỂU ĐỒ 1 — Phân bố số ảnh mỗi nhóm
# Lưu: results/bieudo1_phanbo_label.png
# ============================================================
plt.figure(figsize=(10, 5))
plt.hist(label_counts.values, bins=40, color='steelblue', edgecolor='white')

mean_val = label_counts.mean()
plt.axvline(mean_val, color='red', linestyle='--',
            label=f'Trung bình: {mean_val:.1f} ảnh/nhóm')

plt.title('Biểu đồ 1: Phân bố số ảnh trên mỗi nhóm sản phẩm (label_group)',
          fontsize=13, fontweight='bold')
plt.xlabel('Số ảnh trong nhóm', fontsize=11)
plt.ylabel('Số nhóm sản phẩm', fontsize=11)
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'bieudo1_phanbo_label.png', dpi=150)
plt.show()
print("💾 Đã lưu: results/bieudo1_phanbo_label.png")

print("\n📝 NHẬN XÉT BIỂU ĐỒ 1:")
print(f"  - Phần lớn các nhóm có ít hơn {int(mean_val*2)} ảnh, phân bố lệch phải.")
print(f"  - Có {(label_counts == 2).sum()} nhóm chỉ có đúng 2 ảnh — dataset khá thưa.")
print(f"  - Một số nhóm có tới {label_counts.max()} ảnh — mất cân bằng nhóm rõ rệt.")


In [ ]:
# ============================================================
# CELL 7 (A): BIỂU ĐỒ 2 — Top 15 nhóm nhiều ảnh nhất
# Lưu: results/bieudo2_top15_nhom.png
# ============================================================
top15 = label_counts.head(15)

plt.figure(figsize=(12, 5))
bars = plt.bar(range(len(top15)), top15.values, color='coral', edgecolor='white')

plt.title('Biểu đồ 2: Top 15 nhóm sản phẩm có nhiều ảnh nhất',
          fontsize=13, fontweight='bold')
plt.xlabel('Nhóm sản phẩm (label_group)', fontsize=11)
plt.ylabel('Số lượng ảnh', fontsize=11)
plt.xticks(range(len(top15)),
           [str(l)[:10] + '...' for l in top15.index],
           rotation=45, ha='right')

for bar, val in zip(bars, top15.values):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.2,
             str(val), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'bieudo2_top15_nhom.png', dpi=150)
plt.show()
print("💾 Đã lưu: results/bieudo2_top15_nhom.png")

print("\n📝 NHẬN XÉT BIỂU ĐỒ 2:")
pct = top15.sum() / len(df) * 100
print(f"  - Top 15 nhóm chiếm {top15.sum()} / {len(df)} ảnh ({pct:.1f}% tổng dataset).")
print(f"  - Nhóm đông nhất có {top15.values[0]} ảnh — gấp nhiều lần nhóm trung bình.")
print(f"  - Mất cân bằng này cần lưu ý khi tính Recall@K.")


In [ ]:
# ============================================================
# CELL 8 (B): TIỀN XỬ LÝ & LOAD MODEL
# ============================================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def load_model(device):
    """Load ResNet50 pretrained, bỏ lớp FC, trả về feature extractor."""
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    model = torch.nn.Sequential(*list(model.children())[:-1])  # bỏ FC
    model.eval()
    return model.to(device)

def preprocess_image(img_path):
    """Đọc ảnh và tiền xử lý thành tensor."""
    try:
        img = Image.open(img_path).convert('RGB')
        return transform(img).unsqueeze(0)  # (1, 3, 224, 224)
    except Exception:
        return None

def extract_feature(model, img_tensor, device):
    """Trích xuất vector đặc trưng 2048 chiều."""
    with torch.no_grad():
        feat = model(img_tensor.to(device))
    return feat.squeeze().cpu().numpy()  # (2048,)

model = load_model(DEVICE)
print(f"✅ Load ResNet50 xong! Device: {DEVICE}")


In [ ]:
# ============================================================
# CELL 9 (B): BUILD FAISS INDEX
# Lưu: data/features/index.faiss + data/features/image_paths.pkl
# ============================================================
print(f"\nĐang extract features cho {len(df_subset)} ảnh...")

features = []
valid_paths = []
valid_labels = []

for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Extracting"):
    img_path = DATA_DIR / row['image']
    tensor = preprocess_image(img_path)
    if tensor is None:
        continue
    feat = extract_feature(model, tensor, DEVICE)
    features.append(feat)
    valid_paths.append(str(img_path))
    valid_labels.append(row['label_group'])

features_matrix = np.array(features, dtype='float32')

# L2 normalize — bắt buộc trước khi dùng IndexFlatIP
faiss.normalize_L2(features_matrix)

# Build FAISS index
index = faiss.IndexFlatIP(features_matrix.shape[1])  # 2048-dim
index.add(features_matrix)

# Lưu index và metadata
faiss.write_index(index, str(FEATURES_DIR / 'index.faiss'))
with open(FEATURES_DIR / 'image_paths.pkl', 'wb') as f:
    pickle.dump(valid_paths, f)
with open(FEATURES_DIR / 'image_labels.pkl', 'wb') as f:
    pickle.dump(valid_labels, f)

print(f"\n✅ Build FAISS index xong!")
print(f"   Tổng vector trong index : {index.ntotal}")
print(f"   Feature dimension       : {features_matrix.shape[1]}")
print(f"💾 Đã lưu: data/features/index.faiss")
print(f"💾 Đã lưu: data/features/image_paths.pkl")
print(f"💾 Đã lưu: data/features/image_labels.pkl")


In [ ]:
# ============================================================
# CELL 10 (B): CHẠY BASELINE — TÍNH METRIC
# Lưu: results/baseline_metrics.txt
# ============================================================
def find_similar(query_feat, index, k=5):
    """Tìm k ảnh tương tự nhất, trả về (scores, indices)."""
    q = query_feat.reshape(1, -1).astype('float32')
    faiss.normalize_L2(q)
    scores, indices = index.search(q, k + 1)  # +1 vì kết quả đầu là chính nó
    # Bỏ kết quả đầu tiên (chính ảnh query)
    return scores[0][1:], indices[0][1:]

def precision_at_k(retrieved_labels, query_label, k=5):
    """Tỷ lệ kết quả đúng trong top-K."""
    correct = sum(1 for l in retrieved_labels[:k] if l == query_label)
    return correct / k

def recall_at_k(retrieved_labels, query_label, all_labels, k=5):
    """Tỷ lệ ảnh đúng được tìm thấy trong top-K."""
    total_relevant = sum(1 for l in all_labels if l == query_label) - 1  # trừ chính nó
    if total_relevant == 0:
        return 0.0
    correct = sum(1 for l in retrieved_labels[:k] if l == query_label)
    return correct / total_relevant

def average_precision(retrieved_labels, query_label, k=5):
    """Average Precision cho 1 query."""
    hits, score = 0, 0.0
    for i, label in enumerate(retrieved_labels[:k]):
        if label == query_label:
            hits += 1
            score += hits / (i + 1)
    return score / min(k, sum(1 for l in retrieved_labels if l == query_label) or 1)

# Chọn ảnh query ngẫu nhiên
np.random.seed(42)
query_indices = np.random.choice(len(valid_paths), min(NUM_QUERY, len(valid_paths)), replace=False)

p_scores, r_scores, ap_scores = [], [], []
query_times = []

print(f"Đang chạy baseline với {len(query_indices)} ảnh query...")

for idx in tqdm(query_indices, desc="Baseline"):
    t0 = time.time()
    scores, retrieved_idx = find_similar(features_matrix[idx], index, k=TOP_K)
    query_times.append((time.time() - t0) * 1000)  # ms

    q_label = valid_labels[idx]
    ret_labels = [valid_labels[i] for i in retrieved_idx]

    p_scores.append(precision_at_k(ret_labels, q_label, TOP_K))
    r_scores.append(recall_at_k(ret_labels, q_label, valid_labels, TOP_K))
    ap_scores.append(average_precision(ret_labels, q_label, TOP_K))

# Tổng hợp kết quả
results = {
    'Số ảnh query test'     : len(query_indices),
    'Tổng ảnh trong index'  : index.ntotal,
    f'Precision@{TOP_K}'    : f"{np.mean(p_scores)*100:.2f}%",
    f'Recall@{TOP_K}'       : f"{np.mean(r_scores)*100:.2f}%",
    f'mAP@{TOP_K}'          : f"{np.mean(ap_scores)*100:.2f}%",
    'Thời gian query TB'    : f"{np.mean(query_times):.2f} ms",
    'Thời gian query nhanh nhất': f"{np.min(query_times):.2f} ms",
    'Thời gian query chậm nhất' : f"{np.max(query_times):.2f} ms",
}

print("\n" + "=" * 45)
print("KẾT QUẢ BASELINE")
print("=" * 45)
for k, v in results.items():
    print(f"  {k:<30}: {v}")
print("=" * 45)

# Lưu kết quả
with open(RESULTS_DIR / 'baseline_metrics.txt', 'w', encoding='utf-8') as f:
    f.write("KẾT QUẢ BASELINE — TUẦN 2\n")
    f.write("=" * 45 + "\n")
    for k, v in results.items():
        f.write(f"{k}: {v}\n")
print(f"💾 Đã lưu: results/baseline_metrics.txt")


In [ ]:
# ============================================================
# CELL 11 (C): VISUALIZE — Ảnh query + top-5 kết quả
# Lưu: results/visualize_topk.png
# ============================================================
NUM_SHOW = 4  # số ảnh query để hiển thị

fig = plt.figure(figsize=(16, NUM_SHOW * 3))
fig.suptitle(f'Kết quả tìm kiếm Top-{TOP_K} — Baseline ResNet50 + FAISS',
             fontsize=14, fontweight='bold', y=1.01)

show_indices = query_indices[:NUM_SHOW]

for row_idx, q_idx in enumerate(show_indices):
    _, retrieved_idx = find_similar(features_matrix[q_idx], index, k=TOP_K)
    q_label = valid_labels[q_idx]
    ret_labels = [valid_labels[i] for i in retrieved_idx]

    # Ảnh query
    ax = fig.add_subplot(NUM_SHOW, TOP_K + 1, row_idx * (TOP_K + 1) + 1)
    img = Image.open(valid_paths[q_idx]).convert('RGB')
    ax.imshow(img)
    ax.set_title('QUERY', fontsize=9, fontweight='bold', color='blue')
    ax.axis('off')

    # Top-K kết quả
    for col_idx, (ret_idx, ret_label) in enumerate(zip(retrieved_idx, ret_labels)):
        ax = fig.add_subplot(NUM_SHOW, TOP_K + 1,
                             row_idx * (TOP_K + 1) + col_idx + 2)
        img = Image.open(valid_paths[ret_idx]).convert('RGB')
        ax.imshow(img)
        is_correct = ret_label == q_label
        color = 'green' if is_correct else 'red'
        mark = '✓' if is_correct else '✗'
        ax.set_title(f'#{col_idx+1} {mark}', fontsize=9, color=color)
        ax.axis('off')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'visualize_topk.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Đã lưu: results/visualize_topk.png")


In [ ]:
# ============================================================
# CELL 12 (C): 5 NHẬN XÉT KẾT QUẢ BASELINE
# ============================================================
p_val = np.mean(p_scores) * 100
r_val = np.mean(r_scores) * 100
m_val = np.mean(ap_scores) * 100
t_val = np.mean(query_times)

print("=" * 60)
print("5 NHẬN XÉT KẾT QUẢ BASELINE")
print("=" * 60)
print(f"""
1. Precision@{TOP_K} đạt {p_val:.1f}% — hệ thống đã nhận diện được sự
   tương đồng cơ bản giữa các sản phẩm cùng nhóm, cho thấy
   ResNet50 pretrained có khả năng trích xuất đặc trưng hiệu quả
   ngay cả khi chưa fine-tune trên dataset Shopee.

2. Recall@{TOP_K} chỉ đạt {r_val:.1f}% — do index hiện chỉ có {index.ntotal}
   ảnh, chưa phủ đủ toàn bộ sản phẩm cùng label_group. Khi tăng
   số ảnh trong index, Recall@K dự kiến sẽ cải thiện đáng kể.

3. mAP@{TOP_K} = {m_val:.1f}% — thứ tự xếp hạng còn chưa tốt, ảnh đúng
   chưa luôn xuất hiện ở vị trí đầu tiên. Đây là điểm cần cải
   thiện trong tuần 3 bằng cách tăng tập dữ liệu và xem xét
   fine-tuning mô hình.

4. Thời gian truy vấn trung bình {t_val:.1f} ms/ảnh — FAISS đảm bảo
   tốc độ tìm kiếm gần như realtime ngay cả trên CPU, chứng minh
   tính khả thi của hệ thống trong môi trường thực tế.

5. Một số kết quả sai do ảnh có chữ quảng cáo, logo hoặc viền
   khung màu chiếm diện tích lớn, khiến feature vector bị lệch
   về đặc trưng văn bản/màu sắc thay vì hình dạng sản phẩm gốc.
   Đây là hạn chế đặc thù của dataset Shopee cần xử lý ở tuần 3.
""")


In [ ]:
# ============================================================
# CELL 13 (C): KẾ HOẠCH TUẦN 3
# ============================================================
print("=" * 60)
print("KẾ HOẠCH TUẦN 3")
print("=" * 60)
print("""
Mục tiêu tuần 3: Cải thiện kết quả baseline và hoàn thiện hệ thống.

1. Tăng tập dữ liệu lên 5.000–10.000 ảnh để cải thiện Recall@K.
2. Thử nghiệm VGG16 song song với ResNet50 để so sánh chất lượng
   đặc trưng trích xuất (giải quyết câu hỏi nghiên cứu số 2).
3. Xây dựng giao diện Streamlit: upload ảnh → hiển thị top-10.
4. Đánh giá đầy đủ: Precision@10, Recall@10, mAP@10.
5. Viết báo cáo tuần 3 với so sánh kết quả trước/sau cải thiện.
""")


In [ ]:
# ============================================================
# CELL 14: GHI CHÚ CÔNG CỤ AI
# ============================================================
print("""
⚠️ GHI CHÚ CÔNG CỤ AI:
Nhóm có sử dụng Claude AI và Cursor để hỗ trợ gợi ý cấu trúc
code, pipeline xử lý ảnh và tổ chức notebook. Toàn bộ code
được nhóm đọc, kiểm tra, chạy thử và giải thích lại trước khi
đưa vào báo cáo. Nhóm chịu trách nhiệm về nội dung và kết quả.
""")
